# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoha200/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing Lane 2: Refresh / Content Opportunity Scoring.

I already ran the starter pipeline in ML-01 and saw a random forest beat a hand-written
rule roughly 3x on Precision@50 for flagging pages worth reviewing. That gap is the reason
I want to spend the next 7 weeks here: there's clearly more signal in the data than a
simple rule captures, and a ranked review queue is something a real content team could
use directly, not just a research finding.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zoha200/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows loaded")


30000 rows loaded


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision this improves: which pages should a content editor review first, out of a much
larger backlog than they have time for.

Who acts, and what they do: an SEO/content editor at a FlyRank client. Given a ranked
queue with reason codes, they open the top-ranked pages first and decide whether to
refresh, expand, or leave a page alone.

Cost of a wrong call: two kinds of error matter differently. A false positive (flagging a
healthy page as needing review) costs a wasted editor hour — annoying but cheap. A false
negative (missing a genuinely declining, high-traffic page) costs continued organic traffic
loss on a page that mattered — more expensive. So precision at the top of the queue matters
more than catching every possible case.

In [9]:
declining = df[df["trend_direction"] == "down"]
print(f"Declining pages: {len(declining)} ({100*len(declining)/len(df):.1f}% of all pages)")


Declining pages: 16262 (54.2% of all pages)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [10]:
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
declining_stale = declining[declining["days_since_last_update"] >= 180]

print(f"Declining pages: {len(declining)} ({100*len(declining)/len(df):.1f}% of all pages)")
print(f"Declining AND stale: {len(declining_stale)} ({100*len(declining_stale)/len(declining):.1f}% of declining pages)")

# From ML-01 pipeline run (outputs/model_results.json)
base = 0.240
rf = 0.740
print(f"Baseline Precision@50: {base:.3f} | Random forest Precision@50: {rf:.3f}")

Declining pages: 16262 (54.2% of all pages)
Declining AND stale: 82 (0.5% of declining pages)
Baseline Precision@50: 0.240 | Random forest Precision@50: 0.740


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Three numbers from the starter data that make this lane worth the next 7 weeks:

1. 54.2% of all pages are currently declining — this is not a rare edge case, it's over
   half the inventory. A prioritization tool has a large pool to actually be useful on.
2. Only 0.5% of declining pages are also "stale" (no update in 180+ days) AND visible
   (500+ impressions). A simple staleness rule alone would miss almost all real decline —
   meaning the signal that predicts decline is more complex than "hasn't been updated."
3. In ML-01, a random forest scored 0.740 Precision@50 versus the hand-written rule's
   0.240 — roughly 3x more of its top-50 picks were correct. That's direct evidence a
   learned model captures signal a simple rule misses, on this exact problem.
   

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.